### Initialization

In [ ]:
%matplotlib widget
from stir_plotters import *  # functions for plotting images and projection data
from stir_data_generation import *  # functions to define the NeuroLF scanner geometry and read singles histograms
from scipy.optimize import fsolve
import zipfile
from zenodo_get import download

# whether to do projections with the GPU-based parallelproj library
use_gpu = True

# whether to use the singles-prompts method for randoms estimation (doi:10.1371/journal.pone.0162096)
singles_prompt_method = True

# download the data from zenodo
download("10.5281/zenodo.20392249", file_glob="*.zip")
with zipfile.ZipFile("NeuroLF_Hoffman_Phantom_Dataset_2.zip", "r") as zip_ref:
    zip_ref.extractall("data")

In [ ]:
cylinder_length = 170
cylinder_radius = 108
mu_water = 0.096

attenuation_map = stir.FloatVoxelsOnCartesianGrid(tmpl_proj_data_info, 1)
cylinder = stir.EllipsoidalCylinder(
    cylinder_length,
    cylinder_radius,
    cylinder_radius,
    stir.FloatCartesianCoordinate3D(
        attenuation_map.get_max_z() / 2 * attenuation_map.get_grid_spacing()[1], 0, 0
    ),
)
cylinder.construct_volume(attenuation_map, stir.IntCartesianCoordinate3D(1, 1, 1))
attenuation_map *= mu_water

### Step 1: translate listmode data to sinograms.

In [ ]:
input_proj_data = stir.ProjDataInMemory(tmpl_exam_info, tmpl_proj_data_info)

with open("listmode_file.par", "w") as f:
    f.write(
        f"CListModeDataSAFIR Parameters:=\n  listmode data filename:= data/list-mode/listmode.safir\n  template projection data filename:= NeuroLF-15mm.hs\n  LOR randomization (Gaussian) sigma := 0.0\nEND CListModeDataSAFIR Parameters:="
    )

lm_to_projdata = stir.LmToProjData()
lm_to_projdata.set_store_delayeds(False)
lm_to_projdata.set_output_filename_prefix("projdata_output")
lm_to_projdata.set_template_proj_data_info(tmpl_proj_data_info)
lm_to_projdata.set_time_frame_definitions(time_frame_def)
listmode_data = stir.ListModeData.read_from_file("listmode_file.par")
lm_to_projdata.set_input_data(listmode_data)
lm_to_projdata.set_output_proj_data(input_proj_data)

lm_to_projdata.set_up()
lm_to_projdata.process_data()

acquisition_duration = (
    lm_to_projdata.get_last_processed_lm_rel_time()
)  # read out acquisition duration in seconds

input_proj_data.write_to_file(f"neuroLF_proj_data.hs")
plot_proj(input_proj_data)

### Step 2: construct random sinogram from singles

In [ ]:
coincidence_window = 3.5e-9  # coincidence window in seconds
halflife_f18 = (
    109.7 * 60
)  # the half life of F18 is 109.7 minutes, and we need it in seconds

time_frame_def.set_time_frame(1, 0, acquisition_duration)

singles = np.zeros((number_of_rings, detectors_per_ring))
singles += read_singles_histogram(
    "data/list-mode/singles-histogram.shst",
    time_frame_def.get_start_time(1),
    time_frame_def.get_end_time(1),
)
singles /= time_frame_def.get_end_time(1) - time_frame_def.get_start_time(1)

if singles_prompt_method:
    prompts = stir.FloatArray2D(
        stir.IndexRange2D(
            stir.Int2BasicCoordinate((0, 0)),
            stir.Int2BasicCoordinate((number_of_rings - 1, detectors_per_ring - 1)),
        )
    )
    stir.make_fan_sum_data(prompts, input_proj_data)
    prompts /= time_frame_def.get_end_time(1) - time_frame_def.get_start_time(1)
    prompts = prompts.as_array()

    singles = singles - prompts

efficiencies = stir.FloatArray2D(
    stir.IndexRange2D(
        stir.Int2BasicCoordinate((0, 0)),
        stir.Int2BasicCoordinate((number_of_rings - 1, detectors_per_ring - 1)),
    )
)
efficiencies.fill(singles.flat)

randoms = stir.ProjDataInMemory(tmpl_exam_info, tmpl_proj_data_info)
randoms.fill(1)
decay_corr_factor = stir.decay_correction_factor(
    halflife_f18, time_frame_def.get_start_time(1), time_frame_def.get_end_time(1)
)
double_decay_corr_factor = stir.decay_correction_factor(
    halflife_f18 / 2, time_frame_def.get_start_time(1), time_frame_def.get_end_time(1)
)
scaling_factor = (
    2
    * coincidence_window
    * (time_frame_def.get_end_time(1) - time_frame_def.get_start_time(1))
    * decay_corr_factor
    * decay_corr_factor
    / double_decay_corr_factor
)

if singles_prompt_method:
    lambda_value = fsolve(
        lambda_formula, [1], args=[coincidence_window, np.sum(prompts), np.sum(singles)]
    )[0]
    scaling_factor *= (1 - 2 * lambda_value * coincidence_window) ** 2

stir.multiply_crystal_factors(randoms, efficiencies, scaling_factor)

randoms.write_to_file("neuroLF_random.hs")
plot_proj(randoms)

### Step 3: model attenuation, simulate scatter and compute additive sinogram

In [ ]:
normalisation = stir.ProjDataInMemory.read_from_file(
    "data/Calibration/normalization.hs"
)

# define Hoffman phantom attenuation map
cylinder_length = 170
cylinder_radius = 108
mu_water = 0.096

attenuation_map = stir.FloatVoxelsOnCartesianGrid(tmpl_proj_data_info, 1)
cylinder = stir.EllipsoidalCylinder(
    cylinder_length,
    cylinder_radius,
    cylinder_radius,
    stir.FloatCartesianCoordinate3D(
        attenuation_map.get_max_z() / 2 * attenuation_map.get_grid_spacing()[1], 0, 0
    ),
)
cylinder.construct_volume(attenuation_map, stir.IntCartesianCoordinate3D(1, 1, 1))
attenuation_map *= mu_water
plot_image(attenuation_map)

# compute attenuation correction factors
attenuation_correction_factors = stir.ProjDataInMemory(
    tmpl_exam_info, tmpl_proj_data_info
)
attenuation_correction_factors.fill(1)
if use_gpu:
    forw_proj = stir.ForwardProjectorByBinParallelproj()
else:
    pm = stir.ProjMatrixByBinUsingRayTracing()
    pm.set_use_actual_detector_boundaries(True)
    pm.enable_cache(False)
    forw_proj = stir.ForwardProjectorByBinUsingProjMatrixByBin(pm)
attn_normalisation = stir.BinNormalisationFromAttenuationImage(
    attenuation_map, forw_proj
)
attn_normalisation.set_up(tmpl_exam_info, tmpl_proj_data_info)
attn_normalisation.apply(
    attenuation_correction_factors, forw_proj.get_symmetries_used()
)

In [ ]:
scatter_estimation = stir.ScatterEstimation()

# go through the humble list of settings to be defined
scatter_estimation.set_run_debug_mode(False)
scatter_estimation.set_input_proj_data_sptr(input_proj_data)
scatter_estimation.set_attenuation_image_sptr(attenuation_map)
scatter_estimation.set_normalisation_sptr(
    stir.BinNormalisationFromProjData(normalisation)
)
scatter_estimation.set_attenuation_correction_proj_data_sptr(
    attenuation_correction_factors
)
scatter_estimation.set_background_proj_data_sptr(randoms)

scatter_estimation.set_mask_projdata_filename("neuroLF_mask.hs")
scatter_estimation.set_mask_image_filename("neuroLF_mask.hv")
scatter_estimation.set_max_scale_value(1.1)
scatter_estimation.set_min_scale_value(1.1)
scatter_estimation.set_num_iterations(4)

recon_type = stir.OSMAPOSLReconstruction3DFloat()
recon_type.set_num_subiterations(16)
recon_type.set_num_subsets(16)
recon_type.set_enforce_initial_positivity(True)
obj_func = stir.PoissonLogLikelihoodWithLinearModelForMeanAndProjData3DFloat()
pm = stir.ProjMatrixByBinUsingRayTracing()
pm.enable_cache(False)
pm.set_use_actual_detector_boundaries(True)
proj_pair = stir.ProjectorByBinPairUsingProjMatrixByBin()
proj_pair.set_proj_matrix_sptr(pm)
obj_func.set_zero_seg0_end_planes(True)
obj_func.set_projector_pair_sptr(proj_pair)
recon_type.set_objective_function(obj_func)
recon_type.set_disable_output(True)
scatter_estimation.set_reconstruction_method_sptr(recon_type)
scatter_estimation.set_downsample_scanner(
    True, -1, int(detectors_per_ring / 2)
)  # downsample the scanner to massively speed up the scatter simulation

scatter_estimation.set_output_scatter_estimate_prefix("intermediate_scatter_result")
scatter_estimation.set_export_scatter_estimates_of_each_iteration(True)

simulator = stir.SingleScatterSimulation()
scatter_estimation.set_scatter_simulation_method_sptr(simulator)

scatter_estimation.set_export_scatter_estimates_of_each_iteration(True)

scatter_estimation.set_up()
scatter_estimation.process_data()

scatter = scatter_estimation.get_output()

scatter.write_to_file("neuroLF_scatter.hs")
plot_proj(scatter)

###  Step 4: perform final reconstruction

In [ ]:
target_image = stir.FloatVoxelsOnCartesianGrid(tmpl_proj_data_info, 1)
target_image.fill(1)

empty_proj_data = stir.ProjDataInMemory(tmpl_exam_info, tmpl_proj_data_info)
updated_mult_factors = stir.ProjDataInMemory(attenuation_correction_factors)
updated_mult_factors.sapyb(normalisation, empty_proj_data, empty_proj_data)

additive_sinogram = stir.ProjDataInMemory(scatter)
additive_sinogram.sapyb(
    updated_mult_factors, randoms, updated_mult_factors
)  # (scatter + randoms) * acf * normalisation

plot_proj(updated_mult_factors)
plot_proj(additive_sinogram)

In [ ]:
# now it's time to configure OSMAPOSL and run it
obj_func = stir.PoissonLogLikelihoodWithLinearModelForMeanAndProjData3DFloat()
obj_func.set_recompute_sensitivity(True)
obj_func.set_additive_proj_data_sptr(additive_sinogram)
pd_normalisation = stir.BinNormalisationFromProjData(updated_mult_factors)
obj_func.set_normalisation_sptr(pd_normalisation)
if use_gpu:
    proj_pair = stir.ProjectorByBinPairUsingParallelproj()
else:
    pm = stir.ProjMatrixByBinUsingRayTracing()
    pm.enable_cache(
        False
    )  # this fixes the RAM usage, but makes OSMAPOSL run significantly slower
    pm.set_use_actual_detector_boundaries(True)
    pm.set_num_tangential_LORs(1)
    proj_pair = stir.ProjectorByBinPairUsingProjMatrixByBin()
    proj_pair.set_proj_matrix_sptr(pm)
obj_func.set_zero_seg0_end_planes(True)
obj_func.set_projector_pair_sptr(proj_pair)

recon = stir.OSMAPOSLReconstruction3DFloat()
recon.set_num_subiterations(30)
if use_gpu:
    recon.set_num_subsets(1)
else:
    recon.set_num_subsets(4)
# the following three lines just add debug output
recon.set_disable_output(False)
recon.set_save_interval(10)
recon.set_output_filename_prefix("neuroLF_final_recon_iter")
recon.set_objective_function(obj_func)
recon.set_input_data(input_proj_data)

recon.set_up(target_image)
recon.reconstruct(target_image)

In [ ]:
plot_image(target_image)
stir.write_image_to_file("neuroLF_final_reconstruction.hv", target_image)

### Step 5: Filtering

In [ ]:
# Gaussian filtering
gaussian_image = stir.FloatVoxelsOnCartesianGrid.read_from_file(
    "neuroLF_final_reconstruction.hv"
)
gaussian_filter = stir.SeparableGaussianImageFilter3DFloat()
gaussian_filter.set_fwhms(stir.FloatCartesianCoordinate3D(2.5, 2.5, 2.5))
gaussian_filter.apply(gaussian_image)

stir.write_image_to_file("neuroLF_final_reconstruction_gaussian_2.5.hv", gaussian_image)
plot_image(gaussian_image)